# 🔥 Advanced · Stable Diffusion LoRA / DreamBooth

> 🔥 **Advanced / heavy lab.** Fine-tune a text-to-image diffusion model on a handful of your own images, then generate new ones.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **20–40 min** including downloads. Built on the official **[diffusers (DreamBooth LoRA)](https://huggingface.co/docs/diffusers/training/dreambooth)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

A generative foundation model; the image counterpart to the LLM fine-tuning labs.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — SD 1.5 LoRA / DreamBooth | RTX 4090 / A100 ≥ 24 GB for SDXL |
| **Storage** | SD 1.5 ~ 4–5 GB + LoRA ~ tens of MB | SDXL ~ 14 GB; datasets; 30–60 GB disk |
| **Time** | 400–1500 steps ~ 15–40 min | SDXL LoRA ~ 1–3 h; full fine-tune longer |

**Full pipeline (scale-up):** `accelerate launch train_dreambooth_lora_sdxl.py …`; add prior preservation.

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q diffusers accelerate transformers peft bitsandbytes
!git clone https://github.com/huggingface/diffusers; %cd diffusers/examples/dreambooth
!pip install -q -r requirements.txt

## 1 · A few instance images
Put 5–10 photos of your subject in `data/instance/` (e.g. a specific object or person).

## 2 · Train a LoRA adapter (DreamBooth)

In [ ]:
!accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="data/instance" \
  --instance_prompt="a photo of sks object" \
  --resolution=512 --train_batch_size=1 --gradient_accumulation_steps=1 \
  --learning_rate=1e-4 --lr_scheduler="constant" --max_train_steps=400 \
  --output_dir="sks-lora"

## 3 · Generate with the learned concept

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
pipe.load_lora_weights("sks-lora")
img = pipe("a photo of sks object on the moon, cinematic", num_inference_steps=30).images[0]
img.save("out.png"); img

## Evaluate — image quality & prompt alignment
Text-to-image is scored with **FID** (realism vs a reference set) and **CLIP score** (prompt alignment): `pip install torchmetrics[image]` → `CLIPScore`. For subject/DreamBooth LoRAs, also report **CLIP-I / DINO** subject similarity. Qualitatively, generate a fixed prompt grid before vs after training.

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/<lab>")`.

## How this links to tracks A–D
Generative image models feed the visual tracks:
- **A · Human** generate humans / avatars · **B · 3D** synthesize textures & assets for scenes.

## Troubleshooting & next steps
- Use **prior-preservation** + class images to avoid overfitting; tune steps/LR.
- For style (not subject) use plain **LoRA** text-to-image training.
- Add structural control with **ControlNet** (the next lab); export to a fast runtime (LCM / SDXL-Turbo).